# 🔥 KURE 기반 한국형 번아웃 2단계 분류 모델 v3

## 📋 개요
- **목적**: 일기 텍스트에서 번아웃 감정 분류
- **모델**: KURE (Korean Universal Retrieval Embedding) + Classification Head
- **분류 체계**: 2단계 분류
  - Stage 1: 긍정 vs 부정 (2 클래스)
  - Stage 2: 부정 → 4개 번아웃 카테고리

## 🆕 v3 개선사항 (v2 대비)
1. **데이터셋 확장**: 감성대화 말뭉치 + 웰니스 대화 스크립트 + 한국어 연속적 대화 데이터셋 통합
2. **세션 이어붙이기**: 연속적 대화 데이터에서 동일 감정 연속 발화를 합쳐 일기 유사 텍스트 생성
3. **도메인 다양화**: 상담 맥락(웰니스) 추가로 일기 텍스트와의 도메인 갭 완화

## 📊 데이터 구성 (v3)
| 데이터셋 | 샘플 수 | 특징 |
|---------|--------|------|
| AI Hub 감성대화 말뭉치 (처리본) | ~36,000 | 기존 데이터 |
| 웰니스 대화 스크립트 | ~3,500 | 상담 맥락, 일기 유사 |
| 한국어 연속적 대화 (세션 병합) | ~4,000 | 연속 발화 이어붙이기 |
| **총합** | **~43,500** | |

## 📊 목표 성능
- Stage 1: 88-92% (이전: 88%)
- Stage 2: 55-65% (이전: 47.8%)

---

## 1. 환경 설정

In [ ]:
# GPU 확인
!nvidia-smi

# 필수 라이브러리 설치
!pip install -q sentence-transformers openpyxl

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 디바이스 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. 상수 및 카테고리 정의

In [ ]:
# 카테고리 정의
STAGE1_CATEGORIES = {0: "긍정", 1: "부정"}
STAGE2_CATEGORIES = {
    0: "정서적_고갈",      # Emotional Exhaustion
    1: "좌절_압박",        # Frustration/Pressure
    2: "부정적_대인관계",   # Negative Interpersonal Relations
    3: "자기비하"          # Self-deprecation
}

# 데이터 경로
DATA_PATH      = "/content/drive/MyDrive/Burnout"
DATASET_PATH   = f"{DATA_PATH}/dataset"
PROCESSED_PATH = f"{DATASET_PATH}/processed"

print("✅ 상수 정의 완료")
print(f"   Stage 1: {list(STAGE1_CATEGORIES.values())}")
print(f"   Stage 2: {list(STAGE2_CATEGORIES.values())}")
print(f"   DATA_PATH:      {DATA_PATH}")
print(f"   DATASET_PATH:   {DATASET_PATH}")
print(f"   PROCESSED_PATH: {PROCESSED_PATH}")

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(f"\n📂 경로 확인")
for label, path in [("DATA_PATH", DATA_PATH), ("DATASET_PATH", DATASET_PATH), ("PROCESSED_PATH", PROCESSED_PATH)]:
    exists = os.path.exists(path)
    print(f"   {'✅' if exists else '❌'} {label}: {path}")

if os.path.exists(DATASET_PATH):
    print(f"\n   DATASET 파일 목록: {os.listdir(DATASET_PATH)}")

## 4. 🆕 데이터 전처리

### 세 가지 데이터셋 통합 전략
```
[감성대화 처리본]    → 기존 라벨 그대로 사용 (0~3)
[웰니스 스크립트]   → 카테고리 → 번아웃 라벨 매핑
[연속적 대화]       → 감정 → 번아웃 라벨 매핑 + 세션 이어붙이기
```

### 라벨 체계
- Stage 1: 0=긍정, 1=부정
- Stage 2: 0=정서적_고갈, 1=좌절_압박, 2=부정적_대인관계, 3=자기비하

In [ ]:
# ============================================================
# 4-1. 기존 감성대화 처리본 로드
# ============================================================

existing_train = pd.read_csv(f"{PROCESSED_PATH}/burnout_train_v2.csv")
existing_val   = pd.read_csv(f"{PROCESSED_PATH}/burnout_val_v2.csv")

# Stage 1 라벨: 4개 번아웃 카테고리는 모두 '부정(1)'
existing_train['stage1_label'] = 1
existing_val['stage1_label']   = 1

# Stage 2 라벨: 기존 label 컬럼 그대로
existing_train = existing_train.rename(columns={'label': 'stage2_label'})
existing_val   = existing_val.rename(columns={'label': 'stage2_label'})

print("✅ 기존 감성대화 처리본 로드 완료")
print(f"   Train: {len(existing_train):,}개")
print(f"   Val:   {len(existing_val):,}개")
print(f"   카테고리 분포:\n{existing_train['stage2_label'].value_counts().sort_index().to_string()}")

In [ ]:
# ============================================================
# 4-2. 웰니스 대화 스크립트 전처리
# ============================================================

# 웰니스 카테고리 → 번아웃 라벨 매핑
# 논문 근거: MBI 3요소의 한국형 세분화 기준으로 매핑
WELLNESS_LABEL_MAP = {
    # ── 정서적_고갈 (0): 에너지 소진, 피로, 무기력 ──
    '증상/불면': 0, '증상/불면/생각많음': 0, '증상/불면/스트레스': 0,
    '증상/불면/예민함': 0, '증상/불면/불안감': 0, '증상/불면/피로': 0,
    '감정/힘듦': 0, '감정/힘듦/스트레스': 0, '감정/힘듦/지침': 0,
    '증상/무기력': 0, '증상/무기력/의욕상실': 0, '증상/무기력/은둔': 0,
    '감정/걱정': 0, '감정/걱정/건강염려': 0, '감정/걱정/미래': 0,
    '감정/걱정/증상재발': 0, '감정/걱정/불면': 0, '감정/걱정/경제적문제': 0,
    '감정/불안감': 0, '감정/불안감/증상재발': 0, '감정/불안감/미래': 0,
    '감정/불안감/초조함': 0, '감정/불안감/긴장': 0,
    '감정/우울감': 0, '감정/우울감/증상지속': 0, '감정/우울감/증상재발': 0,
    '감정/우울감/눈물': 0,
    '증상/식욕저하': 0, '증상/식욕저하/체중감소': 0, '증상/식욕저하/불면': 0,
    '증상/기억력저하': 0, '증상/기억력저하/집중력저하': 0,
    '증상/집중력저하': 0, '증상/호흡곤란': 0, '증상/호흡곤란/가슴답답': 0,
    '증상/두통': 0, '증상/두통/불면': 0, '증상/두근거림': 0,
    '증상/두근거림/불면': 0, '증상/피로': 0, '증상/피로/불면': 0,
    '증상/만성피로': 0, '증상/체력저하': 0, '증상/컨디션저조': 0,
    '감정/심란': 0, '감정/눈물': 0, '감정/괴로움': 0,
    '감정/슬픔': 0, '감정/의욕상실': 0, '감정/의욕상실/무기력': 0,
    '감정/절망감': 0, '감정/무력감': 0, '감정/기분저하': 0,
    '감정/멍함': 0, '감정/허탈감': 0, '감정/공포': 0,
    '증상/가슴답답': 0, '감정/두려움': 0, '감정/긴장': 0,
    '증상/체중감소': 0, '증상/어지러움': 0, '증상/소화불량': 0,
    '증상/가슴통증': 0, '증상/힘빠짐': 0, '증상/가슴떨림': 0,

    # ── 좌절_압박 (1): 분노, 불만, 억울함 ──
    '배경/직장': 1, '배경/직장/스트레스': 1, '배경/직장/불만': 1,
    '배경/직장/퇴사': 1, '배경/직장/이직': 1, '배경/직장/반복/이직': 1,
    '배경/직장/과도한업무': 1, '배경/직장/불만/업무': 1, '배경/직장/휴직': 1,
    '배경/직장/복직': 1, '배경/직장/고민/퇴사': 1, '배경/직장/없음/흥미': 1,
    '감정/짜증': 1, '감정/답답': 1, '감정/화': 1, '감정/분노': 1,
    '감정/후회': 1, '감정/불만': 1, '감정/억울함': 1, '감정/좌절': 1,
    '감정/불쾌감': 1, '감정/과민반응': 1, '감정/예민함': 1,
    '감정/초조함': 1, '감정/신경쓰임': 1, '감정/속상함': 1,
    '감정/감정조절이상': 1, '감정/감정조절이상/화': 1,
    '배경/경제적문제': 1, '배경/경제적문제/빚': 1, '배경/경제적문제/가난': 1,
    '배경/사업': 1, '배경/사업/실패': 1, '배경/사업/경제적문제/실패': 1,
    '배경/취업': 1, '배경/취업/힘듦': 1, '배경/취업/준비': 1,
    '배경/대학': 1, '배경/대학/실패': 1, '배경/대학/재수': 1,
    '배경/학업': 1, '배경/학업/부진': 1, '배경/공부': 1, '배경/공부/부진': 1,

    # ── 부정적_대인관계 (2): 대인 갈등, 소외 ──
    '배경/연애': 2, '배경/연애/이별': 2,
    '배경/남편': 2, '배경/남편/갈등': 2, '배경/남편/폭력': 2,
    '배경/남편/무관심': 2, '배경/남편/소통불가': 2, '배경/남편/의심': 2,
    '배경/남자친구': 2, '배경/남자친구/이별': 2, '배경/남자친구/집착': 2,
    '배경/여자친구': 2, '배경/여자친구/이별': 2, '배경/여자친구/관계소원': 2,
    '배경/자녀': 2, '배경/어린시절': 2, '배경/어린시절/가난': 2,
    '배경/친구': 2, '배경/친구/배신': 2, '배경/친구/없음': 2, '배경/친구/관계소원': 2,
    '배경/부모': 2, '배경/부모/갈등': 2, '배경/부모/무관심': 2,
    '배경/부모/이혼': 2, '배경/부모/죽음': 2, '배경/부모/싸움': 2,
    '배경/부모/갈등/아버지': 2, '배경/부모/관계소원': 2,
    '배경/시댁': 2, '배경/시댁/갈등': 2, '배경/시댁/갈등/시어머니': 2,
    '배경/대인관계': 2, '배경/대인관계/갈등': 2, '배경/대인관계/협소': 2,
    '배경/학교': 2, '배경/학교/따돌림': 2, '배경/학교/갈등/선생님': 2,
    '배경/가족': 2, '배경/가족/갈등': 2, '배경/가족/무관심': 2,
    '배경/타인/갈등': 2, '배경/부모/아버지/폭력': 2,
    '증상/은둔': 2, '증상/대인기피': 2, '증상/대화기피': 2,
    '감정/외로움': 2, '감정/고독감': 2, '감정/서운함': 2,
    '감정/배신감': 2, '감정/불신': 2, '감정/미움': 2,

    # ── 자기비하 (3): 자책, 무능감, 부정적 자기인식 ──
    '감정/자살충동': 3, '증상/자살시도': 3, '증상/자해': 3,
    '감정/부정적사고': 3, '증상/피해망상': 3, '증상/피해망상/감시': 3,
    '증상/피해망상/남편': 3, '증상/피해망상/도청': 3,
    '감정/자괴감': 3, '감정/자존감저하': 3, '감정/자신감저하': 3,
    '감정/허무함': 3, '감정/공허감': 3, '감정/비관적': 3,
    '감정/의기소침': 3, '감정/의기소침/자격지심': 3,
    '감정/죄책감': 3, '감정/통제력상실': 3, '감정/창피함': 3,
    '감정/살인욕구': 3, '감정/무미건조': 3, '감정/공허감': 3,
    '증상/이인감': 3, '감정/충격': 3,
}

WELLNESS_PATH = f"{DATASET_PATH}/웰니스 대화 스크립트 데이터셋/웰니스_대화_스크립트_데이터셋.xlsx"
wellness = pd.read_excel(WELLNESS_PATH)
wellness.columns = ['category', 'user_text', 'chatbot_text']

# 유저 발화만 사용 (챗봇 응답 제외)
wellness_filtered = wellness[['category', 'user_text']].copy()
wellness_filtered['stage2_label'] = wellness_filtered['category'].map(WELLNESS_LABEL_MAP)

# 매핑되지 않은 카테고리 제거 (모호함, 부가설명 등)
wellness_filtered = wellness_filtered.dropna(subset=['stage2_label'])
wellness_filtered['stage2_label'] = wellness_filtered['stage2_label'].astype(int)
wellness_filtered['text'] = wellness_filtered['user_text']
wellness_filtered['stage1_label'] = 1  # 모두 부정
wellness_filtered = wellness_filtered[['text', 'stage1_label', 'stage2_label']]

print("✅ 웰니스 대화 스크립트 전처리 완료")
print(f"   총 샘플: {len(wellness_filtered):,}개")
print(f"   카테고리 분포:")
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (wellness_filtered['stage2_label'] == label).sum()
    print(f"     {cat}: {cnt:,}")

In [ ]:
# ============================================================
# 4-3. 연속적 대화 데이터셋 전처리 (세션 이어붙이기)
# ============================================================

# 감정 → 번아웃 라벨 매핑
SEQUENTIAL_EMOTION_MAP = {
    '슬픔': 0,  # 정서적_고갈
    '공포': 0,  # 정서적_고갈
    '분노': 1,  # 좌절_압박
    '혐오': 1,  # 좌절_압박
    '행복': -1, # 긍정 (Stage 1 학습용)
    # 중립, 놀람 → 제외
}

# 감정 레이블 노이즈 정규화 (오타 → 정상값)
EMOTION_NORMALIZE = {
    '분ㄴ': '분노', '분': '분노',
    'ㅈ중립': '중립', 'ㄴ중립': '중립', '중림': '중립',
}

SEQUENTIAL_PATH = f"{DATASET_PATH}/한국어 감정 정보가 포함된 연속적 대화 데이터셋/한국어_연속적_대화_데이터셋.xlsx"
sequential_raw = pd.read_excel(SEQUENTIAL_PATH)

# 컬럼명 정리
sequential_raw.columns = ['session_marker', 'text', 'emotion',
                           'col3', 'col4', '행복', '중립', '슬픔', '공포', '혐오', '분노', '놀람']

# 노이즈 감정 레이블 정규화
sequential_raw['emotion'] = sequential_raw['emotion'].map(
    lambda x: EMOTION_NORMALIZE.get(str(x).strip(), str(x).strip()) if pd.notna(x) else x
)

def merge_sequential_sessions(df, emotion_map, min_len=2, max_group=3):
    """
    세션 이어붙이기:
    - 동일 감정 카테고리의 연속 발화를 묶어 긴 텍스트 생성
    - min_len: 최소 2개 발화 이상 묶인 경우만 사용 (단문 필터링)
    - max_group: 최대 3개 발화까지 묶기 (너무 길어지지 않게)
    """
    merged_samples = []
    group_buffer = []
    group_emotion = None

    def flush_group():
        if len(group_buffer) >= min_len and group_emotion is not None:
            combined_text = ' '.join(group_buffer)
            label = emotion_map.get(group_emotion)
            if label is not None:
                merged_samples.append({
                    'text': combined_text,
                    'emotion': group_emotion,
                    'stage2_label': label,
                    'stage1_label': 0 if label == -1 else 1
                })

    for _, row in df.iterrows():
        marker  = str(row['session_marker']).strip()
        text    = str(row['text']).strip()
        emotion = str(row['emotion']).strip() if pd.notna(row['emotion']) else None

        # 헤더 행 스킵
        if marker == 'dialog #' or text == '발화':
            flush_group()
            group_buffer = []
            group_emotion = None
            continue

        # 유효하지 않은 텍스트 / 매핑 없는 감정 스킵
        if not text or text == 'nan' or emotion not in emotion_map:
            flush_group()
            group_buffer = []
            group_emotion = None
            continue

        # 같은 감정이면 그룹에 추가
        if emotion == group_emotion and len(group_buffer) < max_group:
            group_buffer.append(text)
        else:
            flush_group()
            group_buffer = [text]
            group_emotion = emotion

    flush_group()  # 마지막 그룹
    return pd.DataFrame(merged_samples)


sequential_merged = merge_sequential_sessions(sequential_raw, SEQUENTIAL_EMOTION_MAP)

# 긍정(-1) 샘플 분리 (Stage 1 학습에만 사용)
seq_positive = sequential_merged[sequential_merged['stage2_label'] == -1].copy()
seq_negative = sequential_merged[sequential_merged['stage2_label'] != -1].copy()

seq_negative['stage1_label'] = 1
seq_positive['stage1_label'] = 0
seq_positive['stage2_label'] = -1

print("✅ 연속적 대화 세션 이어붙이기 완료")
print(f"   부정 샘플 (Stage 2용): {len(seq_negative):,}개")
print(f"   긍정 샘플 (Stage 1용): {len(seq_positive):,}개")
print(f"   카테고리 분포 (부정):")
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (seq_negative['stage2_label'] == label).sum()
    print(f"     {cat}: {cnt:,}")

print("\n📝 세션 병합 샘플 예시 (부정):")
for _, row in seq_negative.head(3).iterrows():
    print(f"  [{STAGE2_CATEGORIES[row['stage2_label']]}] {row['text'][:80]}...")

In [ ]:
# ============================================================
# 4-4. 전체 데이터 통합 및 Train/Val 분할
# ============================================================

# 기존 데이터를 공통 포맷으로 변환
existing_train_fmt = existing_train[['text', 'stage1_label', 'stage2_label']].copy()
existing_val_fmt   = existing_val[['text', 'stage1_label', 'stage2_label']].copy()

# 웰니스 + 연속적 부정 → 통합 후 train/val 분할
new_negative = pd.concat([
    wellness_filtered,
    seq_negative[['text', 'stage1_label', 'stage2_label']]
], ignore_index=True)

new_train, new_val = train_test_split(
    new_negative, test_size=0.1, random_state=42,
    stratify=new_negative['stage2_label']
)

# 긍정 샘플 추가 (Stage 1용)
# 기존 데이터에는 긍정 샘플이 없으므로 연속적 대화의 긍정 샘플 활용
pos_train, pos_val = train_test_split(
    seq_positive[['text', 'stage1_label', 'stage2_label']],
    test_size=0.1, random_state=42
)

# 최종 Train / Val 데이터
final_train = pd.concat([
    existing_train_fmt,
    new_train,
    pos_train
], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

final_val = pd.concat([
    existing_val_fmt,
    new_val,
    pos_val
], ignore_index=True).reset_index(drop=True)

# Stage 1 / Stage 2 분리
s1_train = final_train[['text', 'stage1_label']].rename(columns={'stage1_label': 'label'})
s1_val   = final_val[['text', 'stage1_label']].rename(columns={'stage1_label': 'label'})

s2_train = final_train[final_train['stage2_label'] != -1][['text', 'stage2_label']].rename(columns={'stage2_label': 'label'})
s2_val   = final_val[final_val['stage2_label'] != -1][['text', 'stage2_label']].rename(columns={'stage2_label': 'label'})

print("✅ 데이터 통합 완료")
print(f"\n[Stage 1] 긍정 vs 부정")
print(f"   Train: {len(s1_train):,}개")
print(f"   Val:   {len(s1_val):,}개")
print(f"   분포:\n{s1_train['label'].value_counts().to_string()}")
print(f"\n[Stage 2] 4개 번아웃 카테고리")
print(f"   Train: {len(s2_train):,}개")
print(f"   Val:   {len(s2_val):,}개")
print(f"   분포:")
for label, cat in STAGE2_CATEGORIES.items():
    cnt = (s2_train['label'] == label).sum()
    print(f"     {cat}: {cnt:,}")

In [ ]:
# ============================================================
# 4-5. 데이터 저장 (FineTune 노트북에서 재사용)
# ============================================================

os.makedirs(PROCESSED_PATH, exist_ok=True)

s1_train.to_csv(f"{PROCESSED_PATH}/stage1_train_v3.csv", index=False)
s1_val.to_csv(f"{PROCESSED_PATH}/stage1_val_v3.csv",     index=False)
s2_train.to_csv(f"{PROCESSED_PATH}/stage2_train_v3.csv", index=False)
s2_val.to_csv(f"{PROCESSED_PATH}/stage2_val_v3.csv",     index=False)

print("✅ 전처리 데이터 저장 완료")
print(f"   저장 위치: {PROCESSED_PATH}")
print(f"   stage1_train_v3.csv: {len(s1_train):,}개")
print(f"   stage1_val_v3.csv:   {len(s1_val):,}개")
print(f"   stage2_train_v3.csv: {len(s2_train):,}개")
print(f"   stage2_val_v3.csv:   {len(s2_val):,}개")

## 4-6. 🆕 랜덤 혼합 증강 (Random Mixing Augmentation)

### 아이디어
실제 일기는 하나의 감정이 여러 문장에 걸쳐 표현됩니다.
같은 카테고리의 샘플 1~3개를 **무작위로** 합쳐 새로운 복합 텍스트를 생성합니다.

- **모든 데이터셋 적용**: 감성대화 + 웰니스 + 연속적 대화 전부
- **카테고리 유지**: 같은 번아웃 카테고리 내에서만 혼합 → 라벨 오염 없음
- **다양성 증가**: 맥락이 약간 달라도 합치는 것이 허용 (실제 일기도 그렇기 때문)
- **증강 비율**: 원본 데이터 대비 50% 추가 생성

In [ ]:
def random_mix_augmentation(df, label_col='label', augment_ratio=0.5,
                             min_merge=1, max_merge=3, seed=42):
    """
    동일 카테고리 내 무작위 1~3개 샘플 혼합 증강

    Args:
        df: 원본 DataFrame (text, label 컬럼 필요)
        augment_ratio: 원본 대비 생성 비율 (0.5 = 50% 추가)
        min_merge: 최소 혼합 수
        max_merge: 최대 혼합 수
    """
    rng = np.random.default_rng(seed)
    augmented_rows = []

    for label in sorted(df[label_col].unique()):
        texts = df[df[label_col] == label]['text'].tolist()
        n_generate = max(1, int(len(texts) * augment_ratio))

        for _ in range(n_generate):
            k = rng.integers(min_merge, max_merge + 1)
            k = min(k, len(texts))
            indices = rng.choice(len(texts), size=k, replace=False)
            selected = [texts[i] for i in indices]
            combined = ' '.join(selected)
            augmented_rows.append({'text': combined, label_col: label})

    augmented_df = pd.DataFrame(augmented_rows)
    result = pd.concat([df, augmented_df], ignore_index=True).sample(
        frac=1, random_state=seed
    ).reset_index(drop=True)

    return result, augmented_df


# ── Stage 2 Train 데이터 증강 (Val은 그대로 유지) ──
s2_train_aug, s2_aug_only = random_mix_augmentation(
    s2_train, label_col='label', augment_ratio=0.5,
    min_merge=1, max_merge=3, seed=42
)

# ── Stage 1 Train도 부정 샘플 증강 ──
s1_neg_train = s1_train[s1_train['label'] == 1].copy()
s1_pos_train = s1_train[s1_train['label'] == 0].copy()

s1_neg_aug, _ = random_mix_augmentation(
    s1_neg_train, label_col='label', augment_ratio=0.5,
    min_merge=1, max_merge=3, seed=42
)
# 긍정은 소수이므로 더 많이 증강 (균형 보정)
s1_pos_aug, _ = random_mix_augmentation(
    s1_pos_train, label_col='label', augment_ratio=1.0,
    min_merge=1, max_merge=2, seed=42
)

s1_train_aug = pd.concat([s1_neg_aug, s1_pos_aug], ignore_index=True).sample(
    frac=1, random_state=42
).reset_index(drop=True)

print("✅ 랜덤 혼합 증강 완료")
print(f"\n[Stage 1]")
print(f"   원본: {len(s1_train):,}개  →  증강 후: {len(s1_train_aug):,}개")
print(f"   분포:\n{s1_train_aug['label'].value_counts().to_string()}")
print(f"\n[Stage 2]")
print(f"   원본: {len(s2_train):,}개  →  증강 후: {len(s2_train_aug):,}개")
print(f"   카테고리 분포:")
for label, cat in STAGE2_CATEGORIES.items():
    orig = (s2_train['label'] == label).sum()
    aug  = (s2_train_aug['label'] == label).sum()
    print(f"     {cat}: {orig:,} → {aug:,}")

print("\n📝 증강 샘플 예시:")
for _, row in s2_aug_only.head(3).iterrows():
    print(f"  [{STAGE2_CATEGORIES[row['label']]}] {row['text'][:100]}...")

## 5. KURE 임베딩 모델 로드

In [ ]:
print("🔄 KURE 모델 로딩 중...")
kure_model = SentenceTransformer('nlpai-lab/KURE-v1')
kure_model = kure_model.to(device)

test_emb = kure_model.encode("테스트 문장", convert_to_tensor=True)
EMBEDDING_DIM = test_emb.shape[0]
print(f"✅ KURE 로드 완료")
print(f"   Embedding Dim: {EMBEDDING_DIM}")

## 6. 임베딩 생성 (사전 계산)

KURE 인코더는 **Freeze** 상태로 유지하고, 임베딩을 미리 계산하여 학습 속도를 높입니다.

In [ ]:
def generate_embeddings(texts, model, batch_size=64, desc="Embedding"):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i+batch_size]
        with torch.no_grad():
            emb = model.encode(batch, convert_to_tensor=True, show_progress_bar=False)
            embeddings.append(emb.cpu())
    return torch.cat(embeddings, dim=0)

# 증강된 Train 데이터로 임베딩 생성
print("🔄 Stage 1 임베딩 생성 중... (증강 데이터 사용)")
s1_train_emb = generate_embeddings(s1_train_aug['text'].tolist(), kure_model, desc="S1 Train(aug)")
s1_val_emb   = generate_embeddings(s1_val['text'].tolist(),       kure_model, desc="S1 Val")

print("\n🔄 Stage 2 임베딩 생성 중... (증강 데이터 사용)")
s2_train_emb = generate_embeddings(s2_train_aug['text'].tolist(), kure_model, desc="S2 Train(aug)")
s2_val_emb   = generate_embeddings(s2_val['text'].tolist(),       kure_model, desc="S2 Val")

print(f"\n✅ 임베딩 생성 완료")
print(f"   S1 Train(aug): {s1_train_emb.shape}")
print(f"   S1 Val:        {s1_val_emb.shape}")
print(f"   S2 Train(aug): {s2_train_emb.shape}")
print(f"   S2 Val:        {s2_val_emb.shape}")

## 7. 모델 아키텍처 정의

In [ ]:
class BurnoutClassifier(nn.Module):
    def __init__(self, input_dim=1024, hidden_dim=256, num_classes=2, dropout=0.5):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
    def forward(self, x):
        return self.classifier(x)


class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, labels):
        self.embeddings = embeddings
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


print("✅ 모델 클래스 정의 완료")

## 8. Loss Functions (v2와 동일)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        num_classes = inputs.size(-1)
        smooth_targets = torch.zeros_like(inputs).scatter_(1, targets.unsqueeze(1), 1.0)
        smooth_targets = smooth_targets * (1 - self.label_smoothing) + \
                        self.label_smoothing / num_classes
        log_probs = F.log_softmax(inputs, dim=-1)
        probs = torch.exp(log_probs)
        focal_weight = (1 - probs) ** self.gamma
        loss = -focal_weight * smooth_targets * log_probs
        if self.alpha is not None:
            alpha_weight = self.alpha[targets].unsqueeze(1)
            loss = loss * alpha_weight
        return loss.sum(dim=-1).mean()


def compute_class_weights(labels, num_classes):
    counts = np.bincount(labels, minlength=num_classes)
    weights = 1.0 / (counts + 1e-6)
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float32)


print("✅ Loss Functions 정의 완료")

## 9. 학습 함수 (v2와 동일)

In [ ]:
def train_model(
    model, train_emb, train_labels, val_emb, val_labels,
    epochs=50, batch_size=64, lr=1e-3, weight_decay=1e-4,
    patience=7, min_delta=0.001, use_focal_loss=True,
    label_smoothing=0.1, warmup_epochs=5, device='cuda'
):
    train_dataset = EmbeddingDataset(train_emb, train_labels)
    val_dataset   = EmbeddingDataset(val_emb,   val_labels)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader    = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)

    num_classes   = len(np.unique(train_labels))
    class_weights = compute_class_weights(train_labels, num_classes).to(device)
    print(f"   Class Weights: {class_weights.cpu().numpy().round(2)}")

    if use_focal_loss:
        criterion = FocalLoss(gamma=2.0, alpha=class_weights, label_smoothing=label_smoothing)
        print(f"   Loss: FocalLoss (γ=2.0, smoothing={label_smoothing})")
    else:
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
        print(f"   Loss: CrossEntropy (smoothing={label_smoothing})")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / (epochs - warmup_epochs)
        return 0.5 * (1 + np.cos(np.pi * progress))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': [], 'lr': []}
    best_val_f1, best_model_state, patience_counter = 0, None, 0

    print(f"\n{'='*60}")
    print(f"🚀 학습 시작 (epochs={epochs}, batch={batch_size}, lr={lr})")
    print(f"{'='*60}")

    for epoch in range(epochs):
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
        for emb, labels in pbar:
            emb, labels = emb.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(emb)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()
            _, pred = torch.max(logits, 1)
            train_total   += labels.size(0)
            train_correct += (pred == labels).sum().item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        scheduler.step()
        train_acc     = 100 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        all_preds, all_labels_list = [], []

        with torch.no_grad():
            for emb, labels in val_loader:
                emb, labels = emb.to(device), labels.to(device)
                logits = model(emb)
                loss   = criterion(logits, labels)
                val_loss += loss.item()
                _, pred = torch.max(logits, 1)
                val_total   += labels.size(0)
                val_correct += (pred == labels).sum().item()
                all_preds.extend(pred.cpu().numpy())
                all_labels_list.extend(labels.cpu().numpy())

        val_acc      = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        val_f1       = f1_score(all_labels_list, all_preds, average='weighted')
        current_lr   = scheduler.get_last_lr()[0]

        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)
        history['lr'].append(current_lr)

        print(f"Epoch {epoch+1:2d} | Loss: {avg_train_loss:.4f} | "
              f"Train: {train_acc:.1f}% | Val: {val_acc:.1f}% | "
              f"F1: {val_f1:.4f} | LR: {current_lr:.6f}")

        if val_f1 > best_val_f1 + min_delta:
            best_val_f1      = val_f1
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"         ✅ Best F1: {best_val_f1:.4f} (saved)")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n⚠️ Early Stopping at epoch {epoch+1}")
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\n✅ Best Model 복원 (F1: {best_val_f1:.4f})")

    return model, history, {'best_f1': best_val_f1, 'best_acc': max(history['val_acc'])}


print("✅ 학습 함수 정의 완료")

## 10. Stage 1 학습: 긍정 vs 부정

In [ ]:
print("="*60)
print("📊 STAGE 1: 긍정 vs 부정 분류")
print("="*60)

S1_CONFIG = {
    'hidden_dim': 256, 'dropout': 0.5, 'epochs': 50,
    'batch_size': 64, 'lr': 5e-4, 'weight_decay': 1e-4,
    'patience': 7, 'use_focal_loss': False,
    'label_smoothing': 0.1, 'warmup_epochs': 3
}

stage1_model = BurnoutClassifier(
    input_dim=EMBEDDING_DIM, hidden_dim=S1_CONFIG['hidden_dim'],
    num_classes=2, dropout=S1_CONFIG['dropout']
).to(device)

stage1_model, s1_history, s1_best = train_model(
    model=stage1_model,
    train_emb=s1_train_emb, train_labels=s1_train_aug['label'].values,
    val_emb=s1_val_emb,     val_labels=s1_val['label'].values,
    **{k: v for k, v in S1_CONFIG.items() if k not in ['hidden_dim', 'dropout']},
    device=device
)

print(f"\n📈 Stage 1 최종 결과:")
print(f"   Best Val Acc: {s1_best['best_acc']:.2f}%")
print(f"   Best Val F1:  {s1_best['best_f1']:.4f}")

## 11. Stage 2 학습: 4개 번아웃 카테고리

In [ ]:
print("="*60)
print("📊 STAGE 2: 4개 번아웃 카테고리 분류")
print("="*60)

S2_CONFIG = {
    'hidden_dim': 256, 'dropout': 0.5, 'epochs': 80,
    'batch_size': 64, 'lr': 3e-4, 'weight_decay': 1e-4,
    'patience': 10, 'use_focal_loss': True,
    'label_smoothing': 0.15, 'warmup_epochs': 5
}

stage2_model = BurnoutClassifier(
    input_dim=EMBEDDING_DIM, hidden_dim=S2_CONFIG['hidden_dim'],
    num_classes=4, dropout=S2_CONFIG['dropout']
).to(device)

stage2_model, s2_history, s2_best = train_model(
    model=stage2_model,
    train_emb=s2_train_emb, train_labels=s2_train_aug['label'].values,
    val_emb=s2_val_emb,     val_labels=s2_val['label'].values,
    **{k: v for k, v in S2_CONFIG.items() if k not in ['hidden_dim', 'dropout']},
    device=device
)

print(f"\n📈 Stage 2 최종 결과:")
print(f"   Best Val Acc: {s2_best['best_acc']:.2f}%")
print(f"   Best Val F1:  {s2_best['best_f1']:.4f}")

## 12. 결과 시각화

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, (history, best, title) in enumerate([
    (s1_history, s1_best, 'Stage 1'),
    (s2_history, s2_best, 'Stage 2')
]):
    axes[i, 0].plot(history['train_loss'], label='Train')
    axes[i, 0].plot(history['val_loss'],   label='Val')
    axes[i, 0].set_title(f'{title}: Loss')
    axes[i, 0].legend()

    axes[i, 1].plot(history['train_acc'], label='Train')
    axes[i, 1].plot(history['val_acc'],   label='Val')
    axes[i, 1].set_title(f'{title}: Accuracy (Best: {best["best_acc"]:.1f}%)')
    axes[i, 1].legend()

    axes[i, 2].plot(history['val_f1'])
    axes[i, 2].set_title(f'{title}: F1 Score (Best: {best["best_f1"]:.4f})')

plt.tight_layout()
plt.savefig(f"{DATA_PATH}/training_curves_v3.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ 그래프 저장 완료")

## 13. 상세 평가

In [ ]:
def evaluate_model(model, embeddings, labels, categories, stage_name, device='cuda'):
    model.eval()
    dataset = EmbeddingDataset(embeddings, labels)
    loader  = DataLoader(dataset, batch_size=128, shuffle=False)

    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for emb, lab in loader:
            emb = emb.to(device)
            logits = model(emb)
            _, pred = torch.max(logits, 1)
            all_preds.extend(pred.cpu().numpy())
            all_labels_list.extend(lab.numpy())

    print(f"\n{'='*60}")
    print(f"📊 {stage_name} 상세 평가")
    print(f"{'='*60}")
    print(classification_report(
        all_labels_list, all_preds,
        target_names=list(categories.values()), digits=4
    ))

    cm = confusion_matrix(all_labels_list, all_preds)
    print("Confusion Matrix:")
    print(pd.DataFrame(
        cm,
        index=[f"실제_{v}" for v in categories.values()],
        columns=[f"예측_{v}" for v in categories.values()]
    ))
    return all_preds, all_labels_list, cm


s1_preds, s1_labels_eval, _ = evaluate_model(
    stage1_model, s1_val_emb, s1_val['label'].values,
    STAGE1_CATEGORIES, "Stage 1 (긍정/부정)", device
)

s2_preds, s2_labels_eval, _ = evaluate_model(
    stage2_model, s2_val_emb, s2_val['label'].values,
    STAGE2_CATEGORIES, "Stage 2 (번아웃 카테고리)", device
)

## 14. 모델 저장

In [ ]:
torch.save({
    'model_state_dict': stage1_model.state_dict(),
    'embedding_dim': EMBEDDING_DIM, 'hidden_dim': S1_CONFIG['hidden_dim'],
    'num_classes': 2, 'dropout': S1_CONFIG['dropout'],
    'categories': STAGE1_CATEGORIES, 'config': S1_CONFIG,
    'best_metrics': s1_best, 'history': s1_history,
    'data_version': 'v3'
}, f"{DATA_PATH}/stage1_model_v3.pt")

torch.save({
    'model_state_dict': stage2_model.state_dict(),
    'embedding_dim': EMBEDDING_DIM, 'hidden_dim': S2_CONFIG['hidden_dim'],
    'num_classes': 4, 'dropout': S2_CONFIG['dropout'],
    'categories': STAGE2_CATEGORIES, 'config': S2_CONFIG,
    'best_metrics': s2_best, 'history': s2_history,
    'data_version': 'v3'
}, f"{DATA_PATH}/stage2_model_v3.pt")

print(f"✅ Stage 1 저장: {DATA_PATH}/stage1_model_v3.pt")
print(f"✅ Stage 2 저장: {DATA_PATH}/stage2_model_v3.pt")

## 15. 추론 테스트

In [ ]:
def predict_2stage(text, kure, stage1, stage2, device='cuda'):
    stage1.eval(); stage2.eval()
    with torch.no_grad():
        emb = kure.encode(text, convert_to_tensor=True).unsqueeze(0).to(device)
        s1_logits = stage1(emb)
        s1_probs  = F.softmax(s1_logits, dim=-1)[0]
        s1_pred   = torch.argmax(s1_logits, dim=-1).item()

        result = {
            'text': text,
            'stage1': {
                'category': STAGE1_CATEGORIES[s1_pred],
                'confidence': s1_probs[s1_pred].item(),
                'probs': {STAGE1_CATEGORIES[i]: f"{p.item():.1%}" for i, p in enumerate(s1_probs)}
            },
            'stage2': None
        }

        if s1_pred == 1:
            s2_logits = stage2(emb)
            s2_probs  = F.softmax(s2_logits, dim=-1)[0]
            s2_pred   = torch.argmax(s2_logits, dim=-1).item()
            result['stage2'] = {
                'category': STAGE2_CATEGORIES[s2_pred],
                'confidence': s2_probs[s2_pred].item(),
                'probs': {STAGE2_CATEGORIES[i]: f"{p.item():.1%}" for i, p in enumerate(s2_probs)}
            }
    return result


# 일기 스타일 테스트 텍스트
test_texts = [
    "오늘도 야근이었다. 집에 오니 아무것도 하기 싫고 그냥 쓰러지고 싶었다.",
    "팀장이 또 내 앞에서 나를 무시했다. 너무 억울하고 화가 난다.",
    "요즘 들어 출근이 너무 싫다. 사람들 얼굴 보기도 싫고 그냥 다 피하고 싶다.",
    "나는 왜 이것밖에 못 할까. 이러니 아무도 날 인정 안 하지.",
    "오늘 발표가 잘 됐다! 팀장님도 칭찬해 주셔서 기분이 좋았다.",
    "잠을 못 잤더니 온종일 멍했다. 아무것도 집중이 안 되고 너무 지쳤다."
]

print("🧪 일기 스타일 텍스트 테스트")
for text in test_texts:
    r = predict_2stage(text, kure_model, stage1_model, stage2_model, device)
    s1 = r['stage1']
    s2 = r['stage2']
    print(f"\n📝 {r['text'][:50]}...")
    print(f"   Stage 1: {s1['category']} ({s1['confidence']:.1%})")
    if s2:
        print(f"   Stage 2: {s2['category']} ({s2['confidence']:.1%})")
        print(f"   확률: {s2['probs']}")

## 16. 최종 요약

In [ ]:
print("="*70)
print("📋 모델 학습 최종 요약 (v3)")
print("="*70)

print("\n📊 데이터셋 구성")
print("-"*70)
print(f"  AI Hub 감성대화 말뭉치 (처리본):  Train {len(existing_train_fmt):,}개 + Val {len(existing_val_fmt):,}개")
print(f"  웰니스 대화 스크립트:              {len(wellness_filtered):,}개 (train/val 분할)")
print(f"  연속적 대화 세션 병합 (부정):      {len(seq_negative):,}개 (train/val 분할)")
print(f"  연속적 대화 (긍정, Stage 1용):    {len(seq_positive):,}개 (train/val 분할)")
print(f"  Stage 1 총합: Train {len(s1_train):,}개 / Val {len(s1_val):,}개")
print(f"  Stage 2 총합: Train {len(s2_train):,}개 / Val {len(s2_val):,}개")

print("\n🎯 성능 결과")
print("-"*70)
print(f"  Stage 1 (긍정/부정):      Acc = {s1_best['best_acc']:.2f}%,  F1 = {s1_best['best_f1']:.4f}")
print(f"  Stage 2 (번아웃 4분류):   Acc = {s2_best['best_acc']:.2f}%,  F1 = {s2_best['best_f1']:.4f}")

print("\n🆕 v3 추가 사항")
print("-"*70)
print("  1. 웰니스 대화 스크립트: 임상 상담 맥락 추가 (도메인 갭 완화)")
print("  2. 세션 이어붙이기: 연속 발화 2~3개 병합 → 일기 유사 텍스트 생성")
print("  3. 긍정 샘플 추가: Stage 1 학습 데이터 균형 개선")

print("\n" + "="*70)